# Module 05-2: Few-shot 프롬프팅 (솔루션)

## 학습 목표
- Zero-shot, One-shot, Few-shot 기법의 차이를 이해할 수 있다
- `ChatPromptTemplate`에 Human/Assistant 쌍으로 Few-shot 예시를 추가할 수 있다
- FakeLLM으로 이메일 분류 체인을 테스트할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/05-prompt-engineering.md` 섹션 2.2

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: Zero-shot 이메일 분류기

In [ ]:
# Zero-shot 프롬프트 구성
zero_shot = ChatPromptTemplate.from_messages([
    ("system", "이메일을 분류하세요. 카테고리: 문의, 불만, 칭찬, 기타\n반드시 JSON 형식으로만 응답하세요."),
    ("human", "이메일 내용: {email}\n\n카테고리를 JSON으로 답하세요."),
])

zero_shot_llm = FakeLLM(responses={
    "환불": '이메일을 분석하겠습니다. 이 이메일은 문의 카테고리에 해당합니다.',
    "배송": '{"category": "불만"}',
})

zero_shot_chain = zero_shot | zero_shot_llm | StrOutputParser()

print("=== Zero-shot 결과 ===")
for email in ["환불 절차가 어떻게 되나요?", "배송이 2주나 지연되어 화가 납니다."]:
    result = zero_shot_chain.invoke({"email": email})
    print(f"이메일: {email}")
    print(f"결과: {result}\n")

In [ ]:
# 검증 셀
assert zero_shot is not None
test_msgs = zero_shot.format_messages(email="테스트 이메일")
assert len(test_msgs) == 2
assert test_msgs[0].type == "system"
print("실습 1 완료!")

## 실습 2: Few-shot 이메일 분류기 (솔루션)

In [ ]:
# 솔루션
few_shot = ChatPromptTemplate.from_messages([
    ("system", "이메일을 분류하세요. 카테고리: 문의, 불만, 칭찬, 기타\n반드시 JSON 형식으로만 응답하세요."),
    ("human", "이메일 내용: 환불 절차가 어떻게 되나요?"),
    ("assistant", '{"category": "문의"}'),
    ("human", "이메일 내용: 배송이 2주나 지연되어 정말 화가 납니다."),
    ("assistant", '{"category": "불만"}'),
    ("human", "이메일 내용: 친절한 상담 감사합니다. 문제 해결했어요!"),
    ("assistant", '{"category": "칭찬"}'),
    ("human", "이메일 내용: {email}"),
])

few_shot_llm = FakeLLM(responses={
    "환불": '{"category": "문의"}',
    "배송": '{"category": "불만"}',
    "감사": '{"category": "칭찬"}',
    "default": '{"category": "기타"}',
})

few_shot_chain = few_shot | few_shot_llm | StrOutputParser()

print("=== Few-shot 결과 ===")
for email in ["환불 절차가 어떻게 되나요?", "배송이 2주나 지연되어 화가 납니다.", "친절한 상담 감사합니다!"]:
    result = few_shot_chain.invoke({"email": email})
    print(f"이메일: {email}")
    print(f"결과: {result}\n")

In [ ]:
# 검증 셀
assert few_shot is not None
assert few_shot_chain is not None

msgs = few_shot.format_messages(email="테스트")
assert len(msgs) == 8, f"메시지 수가 8개이어야 합니다. 현재: {len(msgs)}개"
assert msgs[0].type == "system"
assert msgs[-1].type == "human"
assert msgs[1].type == "human"
assert msgs[2].type == "ai"
print("실습 2 완료!")

## 실습 3: Zero-shot vs Few-shot 메시지 구조 비교

In [ ]:
# 비교
zero_msgs = zero_shot.format_messages(email="테스트")
few_msgs = few_shot.format_messages(email="테스트")

print(f"Zero-shot: {len(zero_msgs)}개 메시지")
print(f"Few-shot: {len(few_msgs)}개 메시지 ({len(few_msgs) - len(zero_msgs)}개 더 많음)")

In [ ]:
# 검증 셀
assert len(few_msgs) > len(zero_msgs)
assert len(few_msgs) - len(zero_msgs) == 6
print("실습 3 완료!")

## 정리

1. **Zero-shot**: 예시 없이 지시만으로 분류. 간결하지만 형식이 불안정할 수 있음
2. **Few-shot**: Human/Assistant 쌍으로 예시 제공. 형식 준수율 대폭 향상
3. **메시지 순서**: System → (Human+Assistant 쌍 반복) → Human(실제 입력)
4. **트레이드오프**: Few-shot은 토큰을 더 사용하지만 출력 품질이 향상됨